In [0]:
import sys
from importlib.metadata import version

def _version(name):
    try:
        return version(name)
    except Exception:
        return version("mlflow-skinny")

packages = [
    "mlflow",
    "scikit-learn",
    "pandas",
    "numpy",
    "cloudpickle",
]

print("Python:", sys.version)

for package in packages:
    print(f"{package}=={_version(package)}")

assert int(_version("mlflow").split(".")[0]) >= 3, (
    "Use an environment with MLflow 3."
)

In [0]:
CATALOG = "ml_projects"
SCHEMA = "fraud_project"
NAMESPACE = f"{CATALOG}.{SCHEMA}"

FEATURES = [
    "requested_amount",
    "verified_monthly_income",
    "income_discrepancy_ratio",
    "applications_last_90_days",
    "document_inconsistency_count",
    "shared_bank_account_count",
]

spark.conf.set("spark.sql.session.timeZone", "UTC")

display(
    spark.sql(f"SHOW TABLES IN {NAMESPACE}")
)

In [0]:
CATALOG = "ml_projects"
SCHEMA = "fraud_project"

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

AS_OF = "2025-01-01T00:00:00"

applications_table = f"{NAMESPACE}.applications"
labels_table = f"{NAMESPACE}.labels"

applications_version = int(
    spark.sql(f"DESCRIBE HISTORY {applications_table}")
    .orderBy(F.col("version").desc())
    .first()["version"]
)

labels_version = int(
    spark.sql(f"DESCRIBE HISTORY {labels_table}")
    .orderBy(F.col("version").desc())
    .first()["version"]
)

applications_snapshot = (
    spark.read
    .option("versionAsOf", applications_version)
    .table(applications_table)
)

labels_snapshot = (
    spark.read
    .option("versionAsOf", labels_version)
    .table(labels_table)
)

cutoff = F.to_timestamp(F.lit(AS_OF))

eligible = (
    applications_snapshot
    .filter(F.col("application_id").startswith("BASE-"))
    .filter(F.col("application_timestamp") < cutoff)
    .join(
        labels_snapshot.filter(
            F.col("label_available_at") < cutoff
        ),
        on="application_id",
        how="inner",
    )
)

data = (
    eligible.toPandas()
    .sort_values(
        ["application_timestamp", "application_id"]
    )
    .reset_index(drop=True)
)

print("Eligible applications:", len(data))
print("Applications version:", applications_version)
print("Labels version:", labels_version)

display(data.head())

In [0]:
required_features = [
    column
    for column in FEATURES
    if column != "verified_monthly_income"
]

count_features = [
    "applications_last_90_days",
    "document_inconsistency_count",
    "shared_bank_account_count",
]

if data["application_id"].isna().any():
    raise ValueError("Missing application IDs.")

if data["application_id"].duplicated().any():
    raise ValueError("Duplicate application IDs.")

if data[required_features].isna().any().any():
    raise ValueError("Missing required feature values.")

if np.isinf(data[FEATURES].to_numpy(dtype=float)).any():
    raise ValueError("Infinite numerical values.")

if (data[FEATURES] < 0).any().any():
    raise ValueError("Negative feature values.")

if (data["income_discrepancy_ratio"] > 2).any():
    raise ValueError("Discrepancy ratio exceeds the allowed range.")

for column in count_features:
    if not np.equal(
        data[column],
        np.floor(data[column]),
    ).all():
        raise ValueError(f"{column} must contain whole numbers.")

if not data["fraud_label"].isin([0, 1]).all():
    raise ValueError("Fraud labels must be 0 or 1.")

train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)

train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:validation_end].copy()
test = data.iloc[validation_end:].copy()

for name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    if frame["fraud_label"].nunique() != 2:
        raise ValueError(f"{name} must contain both classes.")

    print(
        name,
        "rows:", len(frame),
        "fraud proportion:",
        round(frame["fraud_label"].mean(), 4),
    )

X_train = train[FEATURES]
y_train = train["fraud_label"]

X_validation = validation[FEATURES]
y_validation = validation["fraud_label"]

X_test = test[FEATURES]
y_test = test["fraud_label"]

print("Validation and splitting completed.")

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train)

validation_probability = model.predict_proba(
    X_validation
)[:, 1]

print("Model trained.")
print("Example fraud probabilities:")
print(validation_probability[:10])

In [0]:
from sklearn.metrics import precision_score, recall_score

MAX_REVIEW_RATE = 0.15

thresholds = np.unique(
    np.append(
        validation_probability,
        np.nextafter(
            validation_probability.max(),
            np.inf,
        ),
    )
)

results = []

for threshold in thresholds:
    flagged = validation_probability >= threshold

    results.append({
        "threshold": float(threshold),
        "review_rate": float(flagged.mean()),
        "precision": precision_score(
            y_validation,
            flagged,
            zero_division=0,
        ),
        "recall": recall_score(
            y_validation,
            flagged,
            zero_division=0,
        ),
    })

threshold_results = pd.DataFrame(results)

selected = (
    threshold_results[
        threshold_results["review_rate"] <= MAX_REVIEW_RATE
    ]
    .sort_values(
        ["recall", "precision", "threshold"],
        ascending=[False, False, False],
    )
    .iloc[0]
)

REVIEW_THRESHOLD = float(selected["threshold"])

display(selected.to_frame("selected_value"))

In [0]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve

test_probability = model.predict_proba(X_test)[:, 1]
test_flagged = test_probability >= REVIEW_THRESHOLD

metrics = {
    "test_average_precision": float(
        average_precision_score(y_test, test_probability)
    ),
    "test_roc_auc": float(
        roc_auc_score(y_test, test_probability)
    ),
    "test_precision": float(
        precision_score(y_test, test_flagged, zero_division=0)
    ),
    "test_recall": float(
        recall_score(y_test, test_flagged, zero_division=0)
    ),
    "test_f1": float(
        f1_score(y_test, test_flagged, zero_division=0)
    ),
    "test_brier_score": float(
        brier_score_loss(y_test, test_probability)
    ),
    "test_review_rate": float(test_flagged.mean()),
    "test_fraud_prevalence": float(y_test.mean()),
}

display(pd.DataFrame(
    metrics.items(),
    columns=["metric", "value"],
))

matrix = confusion_matrix(
    y_test,
    test_flagged,
    labels=[0, 1],
)

display(pd.DataFrame(
    matrix,
    index=["Actually legitimate", "Actually fraud"],
    columns=["Not flagged", "Flagged"],
))

observed_rate, mean_prediction = calibration_curve(
    y_test,
    test_probability,
    n_bins=5,
    strategy="quantile",
)

calibration = pd.DataFrame({
    "average_predicted_probability": mean_prediction,
    "observed_fraud_proportion": observed_rate,
})

display(calibration)

In [0]:
import mlflow
import mlflow.pyfunc


class FraudModel(mlflow.pyfunc.PythonModel):
    def __init__(self, pipeline, features, threshold, build_id):
        self.pipeline = pipeline
        self.features = list(features)
        self.threshold = float(threshold)
        self.build_id = build_id

    def predict(self, context, model_input, params=None):
        import numpy as np
        import pandas as pd

        required = ["application_id"] + [
            column
            for column in self.features
            if column != "verified_monthly_income"
        ]

        missing = [
            column
            for column in required
            if column not in model_input.columns
        ]

        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        ids = model_input["application_id"]

        if ids.isna().any() or ids.astype(str).str.strip().eq("").any():
            raise ValueError("application_id cannot be empty.")

        X = (
            model_input
            .reindex(columns=self.features)
            .astype(float)
        )

        required_features = [
            column
            for column in self.features
            if column != "verified_monthly_income"
        ]

        if X[required_features].isna().any().any():
            raise ValueError("Missing required feature values.")

        if np.isinf(X.to_numpy()).any():
            raise ValueError("Infinite feature values.")

        if (X < 0).any().any():
            raise ValueError("Negative feature values.")

        if (X["income_discrepancy_ratio"] > 2).any():
            raise ValueError("Discrepancy ratio exceeds the allowed range.")

        count_columns = [
            "applications_last_90_days",
            "document_inconsistency_count",
            "shared_bank_account_count",
        ]

        for column in count_columns:
            if not np.equal(X[column], np.floor(X[column])).all():
                raise ValueError(f"{column} must contain whole numbers.")

        probability = self.pipeline.predict_proba(X)[:, 1]

        return pd.DataFrame({
            "application_id": ids.to_numpy(),
            "fraud_probability": probability,
            "review_required": probability >= self.threshold,
            "model_build_id": self.build_id,
        })

In [0]:
import sys
import numpy as np
import pandas as pd
import sklearn
import cloudpickle
import mlflow
import mlflow.pyfunc

from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, ColSpec


# 1. Identify where the model and experiment belong.
MODEL_NAME = f"{NAMESPACE}.fraud_model"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

user = spark.sql(
    "SELECT current_user() AS user"
).first()["user"]

EXPERIMENT_PATH = (
    f"/Users/{user}/government-fraud-experiment"
)

mlflow.set_experiment(EXPERIMENT_PATH)


# 2. Define the input and output contract.
input_schema = Schema(
    [ColSpec("string", "application_id")]
    + [
        ColSpec(
            "double",
            column,
            required=(column != "verified_monthly_income"),
        )
        for column in FEATURES
    ]
)

output_schema = Schema([
    ColSpec("string", "application_id"),
    ColSpec("double", "fraud_probability"),
    ColSpec("boolean", "review_required"),
    ColSpec("string", "model_build_id"),
])

signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema,
)


# 3. Read versions from the imported modules.
# This avoids the failing importlib.metadata.version("mlflow") lookup.
dependencies = [
    f"mlflow=={mlflow.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"cloudpickle=={cloudpickle.__version__}",
]

print("Model dependencies:")
for dependency in dependencies:
    print(dependency)


# 4. Prepare a small example of the model's expected input.
input_example = (
    test[["application_id"] + FEATURES]
    .head(5)
    .copy()
)


# 5. Record this training run and save the packaged model.
with mlflow.start_run(run_name="baseline_2025_01") as run:
    RUN_ID = run.info.run_id

    wrapped_model = FraudModel(
        pipeline=model,
        features=FEATURES,
        threshold=REVIEW_THRESHOLD,
        build_id=RUN_ID,
    )

    # Check the wrapper before saving it.
    example_output = wrapped_model.predict(
        context=None,
        model_input=input_example,
    )

    if not example_output["fraud_probability"].between(0, 1).all():
        raise ValueError("Model produced invalid probabilities.")

    mlflow.set_tags({
        "project": "government_fraud",
        "data_type": "synthetic",
        "stage": "baseline",
    })

    mlflow.log_params({
        "model_type": "logistic_regression",
        "as_of": AS_OF,
        "random_state": 42,
        "train_rows": len(train),
        "validation_rows": len(validation),
        "test_rows": len(test),
        "review_threshold": REVIEW_THRESHOLD,
        "validation_review_capacity": MAX_REVIEW_RATE,
        "applications_delta_version": applications_version,
        "labels_delta_version": labels_version,
    })

    mlflow.log_metrics({
        name: float(value)
        for name, value in metrics.items()
    })

    mlflow.log_dict({
        "features": FEATURES,
        "applications_table": applications_table,
        "labels_table": labels_table,
        "as_of": AS_OF,
        "python_version": sys.version,
        "dependencies": dependencies,
        "train_end": str(
            train["application_timestamp"].max()
        ),
        "validation_end": str(
            validation["application_timestamp"].max()
        ),
        "test_end": str(
            test["application_timestamp"].max()
        ),
    }, "training_metadata.json")

    mlflow.log_dict({
        "labels": [0, 1],
        "matrix": matrix.tolist(),
    }, "confusion_matrix.json")

    mlflow.log_dict({
        "bins": calibration.to_dict(orient="records"),
    }, "calibration.json")

    model_info = mlflow.pyfunc.log_model(
        name="fraud_model",
        python_model=wrapped_model,
        signature=signature,
        input_example=input_example,
        pip_requirements=dependencies,
    )


print("Experiment:", EXPERIMENT_PATH)
print("Run ID:", RUN_ID)
print("Saved model URI:", model_info.model_uri)

In [0]:
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=MODEL_NAME,
)

MODEL_VERSION = registered_model.version

print("Registered model:", MODEL_NAME)
print("Registered version:", MODEL_VERSION)
print("Training run:", RUN_ID)

In [0]:
import mlflow
import numpy as np

saved_model = mlflow.pyfunc.load_model(model_info.model_uri)

check_input = test[["application_id"] + FEATURES].head(20).copy()
check_output = saved_model.predict(check_input)

expected_probability = model.predict_proba(check_input[FEATURES])[:, 1]

np.testing.assert_allclose(
    check_output["fraud_probability"].to_numpy(),
    expected_probability,
    rtol=1e-6,
    atol=1e-8
)

assert check_output["application_id"].tolist() == check_input["application_id"].tolist()
assert check_output["model_build_id"].eq(RUN_ID).all()

print("Saved model works. Sample predictions:")
display(check_output)

In [0]:
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=f"{NAMESPACE}.fraud_model"
)

MODEL_VERSION = registered_model.version

print("Registered model:", f"{NAMESPACE}.fraud_model")
print("Model version:", MODEL_VERSION)

In [0]:
import pandas as pd
from mlflow.deployments import get_deploy_client
from pyspark.sql import functions as F

CATALOG = "ml_projects"
SCHEMA = "fraud_project"
NAMESPACE = f"{CATALOG}.{SCHEMA}"
ENDPOINT_NAME = "government-fraud-api"
MODEL_VERSION = "3"

FEATURES = [
    "requested_amount",
    "verified_monthly_income",
    "income_discrepancy_ratio",
    "applications_last_90_days",
    "document_inconsistency_count",
    "shared_bank_account_count",
]

# Берём 10 заявок из уже созданной Delta-таблицы
request_df = (
    spark.table(f"{NAMESPACE}.applications")
    .select("application_id", *FEATURES)
    .limit(10)
    .toPandas()
)

# Формируем запрос для Databricks Serving API.
# dataframe_split сохраняет имена и порядок входных колонок.
request_df = request_df.astype(object).where(pd.notna(request_df), None)

payload = {
    "dataframe_split": {
        "index": list(range(len(request_df))),
        "columns": request_df.columns.tolist(),
        "data": request_df.values.tolist(),
    }
}

# Вызываем уже работающий endpoint
serving_client = get_deploy_client("databricks")
response = serving_client.predict(
    endpoint=ENDPOINT_NAME,
    inputs=payload,
)

# Преобразуем ответ API в pandas DataFrame
predictions_pdf = pd.DataFrame(response["predictions"])

# Создаём таблицу для истории предсказаний, если её ещё нет
predictions_table = f"{NAMESPACE}.predictions"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {predictions_table} (
    application_id STRING,
    prediction_timestamp TIMESTAMP,
    fraud_probability DOUBLE,
    review_required BOOLEAN,
    model_build_id STRING,
    model_version STRING
) USING DELTA
""")

# Добавляем время запроса и версию модели, затем сохраняем результат
(
    spark.createDataFrame(predictions_pdf)
    .withColumn("prediction_timestamp", F.current_timestamp())
    .withColumn("model_version", F.lit(MODEL_VERSION))
    .select(
        "application_id",
        "prediction_timestamp",
        "fraud_probability",
        "review_required",
        "model_build_id",
        "model_version",
    )
    .write.mode("append")
    .saveAsTable(predictions_table)
)

display(
    spark.table(predictions_table)
    .orderBy(F.desc("prediction_timestamp"))
    .limit(10)
)